In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from skimage.morphology import skeletonize, binary_closing, disk
import networkx as nx
import math
import csv
import keras

# --- Configurations --- #
PATCH_SIZE = 256
MASK_THRESHOLD = 0.25
MIN_ROOT_AREA = 250
SAVE_DIR = "temp_debug_roots"
os.makedirs(SAVE_DIR, exist_ok=True)

# --- Utility Functions --- #
def show_image(img, title="", cmap='gray'):
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

# Step 1: Load Image
def load_image(path):
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(f"Image not found or cannot be read: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# Step 2: Extract Petri Dish
def extract_petri_dish(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.medianBlur(gray, 15)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=1000,
                                param1=50, param2=30, minRadius=1000, maxRadius=2000)
    if circles is not None:
        x, y, r = map(int, np.round(circles[0][0]))
        h, w = image_rgb.shape[:2]
        return image_rgb[max(0, y - r):min(h, y + r), max(0, x - r):min(w, x + r)]
    return image_rgb

# Step 3: Patch-wise prediction
def split_into_patches(image, patch_size=256):
    h, w = image.shape[:2]
    patches, coords = [], []
    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patch = image[y:y + patch_size, x:x + patch_size]
            if patch.shape[:2] != (patch_size, patch_size):
                patch = cv2.copyMakeBorder(patch, 0, patch_size - patch.shape[0], 0,
                                           patch_size - patch.shape[1], cv2.BORDER_CONSTANT, value=0)
            patches.append(patch)
            coords.append((y, x))
    return patches, coords, (h, w)

def predict_full_mask(model, image, patch_size=256):
    patches, coords, (H, W) = split_into_patches(image, patch_size)
    full_mask = np.zeros((H, W), dtype=np.float32)
    for patch, (y, x) in zip(patches, coords):
        input_tensor = np.expand_dims(patch / 255.0, axis=0)
        pred = model.predict(input_tensor, verbose=0)[0, :, :, 0]
        h = min(patch_size, H - y)
        w = min(patch_size, W - x)
        full_mask[y:y + h, x:x + w] = pred[:h, :w]
    return (full_mask > MASK_THRESHOLD).astype(np.uint8) * 255

# Step 4: Extract and Assign Roots Left to Right
def extract_roots_from_mask(binary_mask, image_name, min_area=250):
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 5))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel_close)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=8)
    clean_mask = np.zeros_like(opened)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            clean_mask[labels == i] = 255

    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=lambda c: cv2.boundingRect(c)[0])  # sort left to right

    H, W = clean_mask.shape
    slot_edges = np.linspace(0, W, 6)
    plant_paths = [None] * 5

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        cx = x + w // 2
        slot = np.digitize(cx, slot_edges) - 1
        slot = min(max(slot, 0), 4)
        crop = clean_mask[y:y + h, x:x + w]
        path = os.path.join(SAVE_DIR, f"{image_name}_plant_{slot + 1}.png")
        cv2.imwrite(path, crop)
        plant_paths[slot] = path

    return plant_paths

# Step 5: Measure Root Length
def neighbors8(y, x, H, W):
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0: continue
            ny, nx_ = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx_ < W:
                yield ny, nx_

def measure_primary_root_from_array(img):
    if img is None or np.count_nonzero(img) < 100:
        return 0.0
    mask = (img > 127).astype(np.uint8)
    mask = cv2.medianBlur(mask, 3)
    mask = binary_closing(mask.astype(bool), disk(2)).astype(np.uint8)
    skeleton = skeletonize(mask).astype(np.uint8)

    G = nx.Graph()
    H, W = skeleton.shape
    ys, xs = np.where(skeleton)
    for y, x in zip(ys, xs):
        G.add_node((y, x))
        for ny, nx_ in neighbors8(y, x, H, W):
            if skeleton[ny, nx_]:
                dist = math.sqrt((y - ny)**2 + (x - nx_)**2)
                G.add_edge((y, x), (ny, nx_), weight=dist)

    if G.number_of_nodes() < 2:
        return 0.0
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()

    endpoints = [n for n in G.nodes if G.degree[n] == 1]
    best_length = 0.0
    best_score = -1
    for ep1 in endpoints:
        lengths, paths = nx.single_source_dijkstra(G, ep1)
        for ep2 in endpoints:
            if ep2 == ep1 or ep2 not in lengths:
                continue
            path = paths[ep2]
            vertical = abs(path[0][0] - path[-1][0])
            horizontal = abs(path[0][1] - path[-1][1])
            if vertical < horizontal:
                continue
            angle = math.atan2(vertical, horizontal + 1e-5)
            score = math.cos(angle) * 100 + lengths[ep2]
            if score > best_score:
                best_score = score
                best_length = lengths[ep2]
    return best_length

# Full pipeline
def run_pipeline_on_folder(input_dir, model_path, output_csv):
    model = keras.models.load_model(model_path, compile=False)
    rows = []

    for fname in sorted(os.listdir(input_dir)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        image_path = os.path.join(input_dir, fname)
        image_name = os.path.splitext(fname)[0]
        rgb = load_image(image_path)
        petri = extract_petri_dish(rgb)
        binary_mask = predict_full_mask(model, petri)
        plant_paths = extract_roots_from_mask(binary_mask, image_name)

        for i in range(5):
            plant_id = f"{image_name}_plant_{i + 1}"
            if plant_paths[i] is None:
                length = 0
            else:
                mask = cv2.imread(plant_paths[i], cv2.IMREAD_GRAYSCALE)
                length = int(round(measure_primary_root_from_array(mask)))
            rows.append([plant_id, length])

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Plant ID", "Length (px)"])
        writer.writerows(rows)

    return output_csv

# Trigger pipeline
output_file = run_pipeline_on_folder(
    input_dir="client_data/Kaggle",
    model_path="MoaliJaberii_244437_unet_model_256px.keras",
    output_csv="root_lengths_from_debug_pipeline.csv"
)
output_file


2026-01-14 11:20:48.824986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
2026-01-14 11:20:53.293557: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-01-14 11:20:53.293598: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-01-14 11:20:53

'root_lengths_from_debug_pipeline.csv'